In [1]:
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.linear_model import LogisticRegression
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold


import xgboost as xgb
from sklearn.preprocessing import LabelEncoder

# 1. Load data
* Need to read the Prior.csv (mine d'or)

In [2]:
## Loading the data
train_df = pd.read_csv('Train.csv')
test_df = pd.read_csv('Test.csv')

prior_df = pd.read_csv('Prior.csv')

#y_true = pd.read_csv('reference.csv')['Target_AUC']


In [3]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13536 entries, 0 to 13535
Data columns (total 17 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   ID                       13536 non-null  object
 1   farmer_name              13536 non-null  object
 2   training_day             13536 non-null  object
 3   gender                   13536 non-null  object
 4   registration             13536 non-null  object
 5   age                      13536 non-null  object
 6   group_name               13536 non-null  object
 7   belong_to_cooperative    13536 non-null  int64 
 8   county                   13536 non-null  object
 9   subcounty                13536 non-null  object
 10  ward                     13536 non-null  object
 11  adopted_within_07_days   13536 non-null  int64 
 12  adopted_within_90_days   13536 non-null  int64 
 13  adopted_within_120_days  13536 non-null  int64 
 14  has_topic_trained_on     13536 non-nul

In [4]:
# =========================
# FEATURE ENGINEERING START
# =========================

# Convert dates
train_df["training_day"] = pd.to_datetime(train_df["training_day"], errors="coerce")
test_df["training_day"] = pd.to_datetime(test_df["training_day"], errors="coerce")
prior_df["training_day"] = pd.to_datetime(prior_df["training_day"], errors="coerce")

# Date features: day, month, weekday
for df in [train_df, test_df]:
    df["day"] = df["training_day"].dt.day
    df["month"] = df["training_day"].dt.month
    df["weekday"] = df["training_day"].dt.weekday

# Topic count feature
for df in [train_df, test_df]:
    df["topic_count"] = df["topics_list"].astype(str).apply(lambda x: len(x.split(",")))

# Farmer experience
farmer_counts = prior_df.groupby("farmer_name").size()
train_df["farmer_training_count"] = train_df["farmer_name"].map(farmer_counts)
test_df["farmer_training_count"]  = test_df["farmer_name"].map(farmer_counts)

# Farmer adoption rate
farmer_adoption = prior_df.groupby("farmer_name")["adopted_within_90_days"].mean()
train_df["farmer_adoption_rate"] = train_df["farmer_name"].map(farmer_adoption)
test_df["farmer_adoption_rate"]  = test_df["farmer_name"].map(farmer_adoption)

# Trainer performance
trainer_adoption = prior_df.groupby("trainer")["adopted_within_90_days"].mean()
train_df["trainer_adoption_rate"] = train_df["trainer"].map(trainer_adoption)
test_df["trainer_adoption_rate"]  = test_df["trainer"].map(trainer_adoption)
mean_trainer = train_df["trainer_adoption_rate"].mean()

# County performance
county_adoption = prior_df.groupby("county")["adopted_within_90_days"].mean()
train_df["county_adoption_rate"] = train_df["county"].map(county_adoption)
test_df["county_adoption_rate"]  = test_df["county"].map(county_adoption)
mean_county = train_df["county_adoption_rate"].mean()

# Fill missing values properly (plus compatible pandas 3.0)
for col, fill_value in [
    ("farmer_training_count", 0),
    ("farmer_adoption_rate", 0),
    ("trainer_adoption_rate", mean_trainer),
    ("county_adoption_rate", mean_county)
]:
    train_df[col] = train_df[col].fillna(fill_value)
    test_df[col]  = test_df[col].fillna(fill_value)

# =========================
# FEATURE ENGINEERING END
# =========================

# 2. Data wrangling

In [5]:
target_col = 'adopted_within_07_days'
train_df[target_col] = train_df[target_col].astype(int)

In [6]:
split_summary = pd.DataFrame({
    "set": ["train"],
    "rows": [len(train_df)],
    "positives": [train_df[target_col].sum()],
})
split_summary["pos_rate"] = split_summary["positives"] / split_summary["rows"]

split_summary


,set,rows,positives,pos_rate
0,train,13536,153,0.011303


In [7]:
## Basic model with some selected features
base_features = [
    "gender",
    "registration",
    "age",
    "trainer",
    "belong_to_cooperative",
    "county",
    "subcounty",
    "ward",

    # new features
    "day",
    "month",
    "weekday",
    "topic_count",
    "farmer_training_count",
    "farmer_adoption_rate",
    "trainer_adoption_rate",
    "county_adoption_rate",
]


In [8]:

from sklearn.model_selection import train_test_split

feature_cols = base_features

X = train_df[feature_cols]
y = train_df[target_col]

# X_train = train_df[feature_cols]
# y_train = train_df[target_col]


X_train,X_val,y_train,y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

X_test = test_df[feature_cols]

## Make the prediction


In [9]:
from catboost import CatBoostClassifier

cat_features = [
    "gender",
    "registration",
    "trainer",
    "county",
    "subcounty",
    "ward",
    "age"
]

model = CatBoostClassifier(
    iterations=3000,
    learning_rate=0.02,
    depth=6,
    eval_metric="Logloss",
    random_seed=42,
    verbose=200
)

model.fit(
    X_train,
    y_train,
    cat_features=cat_features,
    eval_set=(X_val, y_val),
    early_stopping_rounds=200
)

y_pred_prob = model.predict_proba(X_val)[:,1]

0:	learn: 0.6488250	test: 0.6487638	best: 0.6487638 (0)	total: 81.5ms	remaining: 4m 4s
200:	learn: 0.0299930	test: 0.0361279	best: 0.0361279 (200)	total: 1.74s	remaining: 24.3s
400:	learn: 0.0239731	test: 0.0345633	best: 0.0345370 (399)	total: 3.3s	remaining: 21.4s
600:	learn: 0.0208599	test: 0.0343255	best: 0.0343203 (598)	total: 4.94s	remaining: 19.7s
800:	learn: 0.0186112	test: 0.0346357	best: 0.0343152 (602)	total: 6.57s	remaining: 18s
Stopped by overfitting detector  (200 iterations wait)

bestTest = 0.03431517705
bestIteration = 602

Shrink model to first 603 iterations.


In [10]:
## Evaluating some metrics
# ----------------------------
#  Metrics: PR AUC + ROC AUC + Recall@K
# ----------------------------
def recall_at_k(y_true: pd.Series, y_scores: np.ndarray, k_frac: float) -> float:
    k = int(np.ceil(len(y_true) * k_frac))
    order = np.argsort(-y_scores)  # descending
    topk = order[:k]
    return float(y_true.iloc[topk].sum() / y_true.sum()) if y_true.sum() > 0 else np.nan

In [11]:
pr_auc = average_precision_score(y_val, y_pred_prob)
roc_auc = roc_auc_score(y_val, y_pred_prob)
recall_5 = recall_at_k(y_val, y_pred_prob, 0.05)
recall_10 = recall_at_k(y_val, y_pred_prob, 0.10)
recall_20 = recall_at_k(y_val, y_pred_prob, 0.20)

# Naive baselines on this test set
prevalence_test = y_val.mean()
naive_pr_auc = prevalence_test

results = pd.DataFrame({
    "metric": [
        "PR AUC (Average Precision)",
        "ROC AUC",
        "Recall@5%",
        "Recall@10%",
        "Recall@20%",
        "Naive PR AUC (prevalence)",
    ],
    "value": [pr_auc, roc_auc, recall_5, recall_10, recall_20, naive_pr_auc]
})


print("\nSplit summary:")
display(split_summary)
print("\nResults:")
display(results)


Split summary:


,set,rows,positives,pos_rate
0,train,13536,153,0.011303



Results:


,metric,value
0,PR AUC (Average Precision),0.335260
1,ROC AUC,0.974020
2,Recall@5%,0.741935
3,Recall@10%,0.935484
4,Recall@20%,1.000000
5,Naive PR AUC (prevalence),0.011448


In [12]:
y_pred_prob

array([0.00016138, 0.00356905, 0.00021378, ..., 0.00012722, 0.00024589,
       0.00209248])

In [13]:
y_test_pred_prob=model.predict_proba(X_test)[:, 1]

## Make submission file
ss = pd.read_csv('SampleSubmission.csv')
ss['Target_LogLoss'] = y_test_pred_prob
ss['Target_AUC'] = y_test_pred_prob

ss.to_csv('BenchmarkSub.csv', index=False)


,ID,Target_LogLoss,Target_AUC
0,ID_LEG1GM,0.000477,0.000477
1,ID_1UKOKW,0.001156,0.001156
2,ID_U5H2YK,0.048325,0.048325
3,ID_55957A,0.034378,0.034378
4,ID_N1AC0A,0.000479,0.000479


In [15]:
from sklearn.metrics import log_loss

In [16]:
## Compute the log-loss
loss = log_loss(y_val, y_pred_prob)
print("LogLoss:", loss)


LogLoss: 0.03431517704660782


In [17]:
train_df.columns

Index(['ID', 'farmer_name', 'training_day', 'gender', 'registration', 'age',
       'group_name', 'belong_to_cooperative', 'county', 'subcounty', 'ward',
       'adopted_within_07_days', 'adopted_within_90_days',
       'adopted_within_120_days', 'has_topic_trained_on', 'trainer',
       'topics_list', 'day', 'month', 'weekday', 'topic_count',
       'farmer_training_count', 'farmer_adoption_rate',
       'trainer_adoption_rate', 'county_adoption_rate'],
      dtype='object')